In [ ]:
!nvidia-smi

In [ ]:
from unsloth import FastModel
from tqdm.auto import tqdm
from unsloth.chat_templates import get_chat_template
import random
import json
import os

from MonsterNameGenerator import MarkovMonsterNameGenerator
from textGenerateUtils import generate_scientific_name, generate_prompt, generate_description

In [ ]:
text_model_name = "unsloth/gemma-3-27b-it"
chat_template = "gemma-3"

monster_name_filepath = 'monsterNames.txt'

output_dir = "generated_texts"
os.makedirs(output_dir, exist_ok=True)

In [ ]:
model, tokenizer = FastModel.from_pretrained(
    model_name = text_model_name,
    max_seq_length = 2048,
    load_in_4bit = True,
    load_in_8bit = False,
    full_finetuning = False,
    device_map = "auto"
)

In [ ]:
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers     = False,
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,

    r = 8,
    lora_alpha = 8,
    lora_dropout = 0,
    bias = "none"
)

In [ ]:
tokenizer = get_chat_template(
    tokenizer,
    chat_template = chat_template,
)

In [ ]:
name_generator = MarkovMonsterNameGenerator(n=2)
name_generator.train_from_file(monster_name_filepath)

In [ ]:
fields = [
    "杉林", "古代林", "畑", "草むら", "花畑", "密林", "水没林","ジャングル","峠","山の麓","樹海","竹林","森","霧の森","熱帯雨林","サバンナ","桜並木","果樹園",
    "洞窟", "鍾乳洞","谷底","岩石地帯","鉱山","荒野","岩の中",
    "雪原","凍土","氷河",
    "旧市街地", "化学工場跡地","都市の下水道", "古城","都市部","廃工場","地下鉄廃線","空中都市",
    "大砂漠", "オアシス",
    "海", "深海", "浅瀬", "砂浜", "汽水域", "川底","孤島","海底遺跡","湖","潮溜まり","地下水路","滝","沈没船","サンゴ礁",
    "成層圏","惑星中心部","溶岩地帯",
    "モンスターの体内"
]
spicies = [
    "生物",
    "鳥",
    "虫",
    "植物",
    "花",
    "草",
    "木",
    "キノコ",
    "魚",
    "爬虫類",
    "哺乳類",
    "両生類",
    "巨大生物",
    "小型生物",
    "草食動物",
    "肉食動物",
    "寄生生物",
    "絶滅危惧種",
    "甲殻類",
    "貝",
    "群生生物",
    "原始生物",
    "人工生命",
    "分類不明の生物"
]

In [ ]:
for j in tqdm(range(25)):
    name = name_generator.generate()
    field = random.choice(fields)
    if field == "モンスターの体内":
        field = name_generator.generate()+"の体内"
    spicy = random.choice(spicies)
    if spicy in ("貝","草","鳥","魚") and random.randint(0,1)==1:
        name = name + spicy
    target = "{0}にて観測される架空の{1}「{2}」".format(field, spicy, name)
    print(target)
    description = generate_description(target, model, tokenizer)
    prompt = generate_prompt(target, description, model, tokenizer)
    scientific_name = generate_scientific_name(target, description, model, tokenizer)

    data = {
        "name": name,
        "target": target,
        "description": description,
        "prompt": prompt,
        "scientific_name": scientific_name,
    }
    with open(os.path.join(output_dir, f"{j}-{name}.json"), "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)